# Exploratory Data Analysis (EDA)
## Public Compliance Data Analysis - MBA Thesis

**Objective:** Explore the Gold layer datasets at **municipality level** (5,570 records, 1 row per Brazilian municipality) to understand:
- Data distributions and summary statistics
- Missing values and data quality
- Initial patterns and relationships
- Regional variations in compliance and socioeconomic indicators

**Grain note.** The main analytical dataset used here is `analysis_compliance_municipality` (one row per municipality, N=5,570). The previous state-level rollup (`analysis_compliance`, N=27) had too few observations for meaningful correlation/regression analysis. To preserve geographic context, `state_code` / `state_name` / `region_code` / `region_name` are kept as identifier columns and one-hot encoded into dummy variables (`is_region_*`, `is_state_*`) the same way the state-level dataset encoded region.

In [ ]:
# --- AUTO-GENERATED DEPENDENCY INSTALL ---
# Installs all project dependencies on first run (Colab, fresh environments, etc).
# Idempotent: pip skips anything already installed.
# To regenerate this cell, run: python scripts/inject_pip_install.py

import subprocess
import sys
from pathlib import Path

_req = Path.cwd().parent / "requirements.txt"
if not _req.exists():
    _req = Path.cwd() / "requirements.txt"

if _req.exists():
    print(f"Installing dependencies from {_req.name if _req.exists() else "requirements.txt"} ...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(_req)])
    print("Dependencies ready.")
else:
    print("requirements.txt not found. Install manually: pip install -r requirements.txt")


## Step-by-step (Aula style)

1. Packages and environment setup
2. Reproducibility
3. Data loading
4. Analysis blocks
5. Summary and interpretation


# Packages


In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Try local loader first, fallback to S3 if available
from src.analysis.local_data_loader import LocalGoldDataLoader as GoldDataLoader

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 2)
pd.set_option('display.float_format', '{:.2f}'.format)


In [ ]:
import matplotlib as mpl
mpl.rcParams['axes.formatter.useoffset'] = False
mpl.rcParams['axes.formatter.limits'] = (-99, 99)


# Reproducibility


In [ ]:
import os
import random

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

print(f"Reproducibility seed fixed at {SEED}")


In [ ]:
import json as _json

_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}

S3_BUCKET_NAME = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))
AWS_PROFILE = os.environ.get('AWS_PROFILE', _rtcfg.get('aws', {}).get('profile', None))


## 1. Load Data

In [ ]:
loader = GoldDataLoader()

print("Available datasets:")
for dataset in loader.list_available_datasets():
    print(f"  • {dataset}")


In [ ]:
datasets = loader.load_all()

df_muni = datasets.get('municipality_socioeconomic')
df_state = datasets.get('state_summary')
df_sanctions = datasets.get('sanctions_summary')

# Primary analytical dataset: one row per municipality (N ~= 5,570).
df_analysis = datasets.get('analysis_compliance_municipality')

# --- Add one-hot region and state dummies (not present in the muni dataset). ---
# These mirror the is_norte / is_nordeste / ... columns that existed on the
# state-level dataset -- keeping the same human-readable names so any downstream
# notebook that referenced those columns continues to work.
# Kept as Int64 (0/1) for consistency with the pre-existing is_* convention and
# for direct use as regression features in notebooks 02 / 03.
REGION_NAME_TO_DUMMY = {
    'Norte': 'is_norte',
    'Nordeste': 'is_nordeste',
    'Sudeste': 'is_sudeste',
    'Sul': 'is_sul',
    'Centro-Oeste': 'is_centro_oeste',
}
for rname, col in REGION_NAME_TO_DUMMY.items():
    df_analysis[col] = (df_analysis['region_name'] == rname).astype('Int64')
REGION_DUMMY_COLS = list(REGION_NAME_TO_DUMMY.values())

# State dummies: one per state, named by 2-digit IBGE state code.
state_dummies = pd.get_dummies(df_analysis['state_code'], prefix='is_state').astype('Int64')
df_analysis = pd.concat([df_analysis, state_dummies], axis=1)
STATE_DUMMY_COLS = list(state_dummies.columns)

print(f"\n✅ Loaded {len(datasets)} datasets")
print(f"   Main analytical dataset: analysis_compliance_municipality")
print(f"   Rows: {len(df_analysis):,} municipalities, {df_analysis['state_code'].nunique()} states, {df_analysis['region_code'].nunique()} regions")
print(f"   Region dummies added ({len(REGION_DUMMY_COLS)}): {REGION_DUMMY_COLS}")
print(f"   State dummies added ({len(STATE_DUMMY_COLS)}): first 3 = {STATE_DUMMY_COLS[:3]} ... last = {STATE_DUMMY_COLS[-1]}")

## 2. Dataset Overview

In [ ]:
print("=" * 80)
print("ANALYSIS COMPLIANCE MUNICIPALITY DATASET")
print("=" * 80)
print(f"Shape: {df_analysis.shape}  (rows = municipalities, cols = features + dummies)")
print(f"\nDtypes (non-dummy columns only):")
print(df_analysis.drop(columns=REGION_DUMMY_COLS + STATE_DUMMY_COLS).dtypes)
print(f"\nMemory Usage: {df_analysis.memory_usage(deep=True).sum() / 1024 / 1024:.2f} MB")

In [ ]:
# Preview a handful of columns (hiding the 32 region+state dummies).
preview_cols = [c for c in df_analysis.columns if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
df_analysis[preview_cols].head(10)

In [ ]:
df_analysis.info(verbose=False)

## 3. Descriptive Statistics

In [ ]:
# Describe only the analytical features, excluding the 32 one-hot dummies
# (dummies are summarized separately below).
numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns
analytical_numeric = [c for c in numeric_cols if c not in REGION_DUMMY_COLS + STATE_DUMMY_COLS]
df_analysis[analytical_numeric].describe().T

### Key Metrics Summary

In [ ]:
# Population-weighted averages for rates. A straight mean across 5,570
# municipalities would treat a 500-inhabitant town and Sao Paulo city equally,
# which is statistically misleading for rate and average indicators.
_pop = df_analysis['population_2022']
_lit_mask = df_analysis['literacy_rate_2022'].notna()
_inc_mask = df_analysis['avg_income_2022'].notna()

total_pop = int(_pop.sum())
total_sanc = int(df_analysis['n_sanctions'].sum())

summary = pd.DataFrame({
    'Total Municipalities': [len(df_analysis)],
    'Total States': [int(df_analysis['state_code'].nunique())],
    'Total Regions': [int(df_analysis['region_code'].nunique())],
    'Total Population (2022)': [total_pop],
    'Total Sanctions': [total_sanc],
    'Sanctions/100k (national, pop-weighted)': [round(total_sanc / total_pop * 100_000, 2)],
    'Literacy % (pop-weighted)': [round(np.average(df_analysis.loc[_lit_mask, 'literacy_rate_2022'], weights=_pop[_lit_mask]), 2)],
    'Avg Income BRL (pop-weighted)': [round(np.average(df_analysis.loc[_inc_mask, 'avg_income_2022'], weights=_pop[_inc_mask]), 2)],
})

summary.T

**Dummy variable summary.** For the one-hot region and state dummies the mean equals the proportion of municipalities in that region / state (i.e. `df['is_region_1'].mean()` is the share of munis in the Norte region). A short summary is shown below so we do not flood the main `.describe()` table with 32 extra rows.

In [ ]:
dummy_stats = pd.DataFrame({
    'Count (=1)': df_analysis[REGION_DUMMY_COLS + STATE_DUMMY_COLS].sum(),
    'Share of munis %': (df_analysis[REGION_DUMMY_COLS + STATE_DUMMY_COLS].mean() * 100).round(2),
}).sort_values('Count (=1)', ascending=False)
print(f"Total dummies: {len(dummy_stats)} (5 regions + {len(STATE_DUMMY_COLS)} states)")
dummy_stats.head(10)

## 4. Missing Values Analysis

In [ ]:
missing = df_analysis.isnull().sum()
missing_pct = (missing / len(df_analysis)) * 100

missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2),
}).sort_values('Missing Count', ascending=False)

missing_df[missing_df['Missing Count'] > 0]

## 5. Distribution Analysis

### 5.1 Target Variable: Sanctions per 100k

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(df_analysis['sanctions_per_100k'], bins=20, edgecolor='black', alpha=0.7)
axes[0].set_title('Distribution of Sanctions per 100k', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Sanctions per 100k Population')
axes[0].set_ylabel('Frequency')
axes[0].axvline(df_analysis['sanctions_per_100k'].mean(), color='red', linestyle='--', label='Mean')
axes[0].axvline(df_analysis['sanctions_per_100k'].median(), color='green', linestyle='--', label='Median')
axes[0].legend()

axes[1].boxplot(df_analysis['sanctions_per_100k'])
axes[1].set_title('Boxplot: Sanctions per 100k', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Sanctions per 100k Population')

from scipy import stats
stats.probplot(df_analysis['sanctions_per_100k'], dist="norm", plot=axes[2])
axes[2].set_title('Q-Q Plot: Sanctions per 100k', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print(f"Skewness: {df_analysis['sanctions_per_100k'].skew():.3f}")
print(f"Kurtosis: {df_analysis['sanctions_per_100k'].kurtosis():.3f}")


### 5.2 Socioeconomic Indicators

**Note on Log Transformation:** Population data is typically highly right-skewed — a few states have extremely large populations while most have much smaller values. The logarithmic transformation (log_population) addresses this by:
1. **Normalizing the distribution** — making it more symmetric for valid statistical analysis
2. **Reducing the influence of extreme values** — preventing large populations from disproportionately affecting correlations and regressions
3. **Enabling percentage-based interpretation** — in regression models, changes represent proportional effects

The histograms below compare the raw population distribution (skewed) with the log-transformed version (more normal).

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

axes[0, 0].hist(df_analysis['literacy_rate_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='skyblue')
axes[0, 0].set_title('Literacy Rate 2022 Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[0, 0].set_xlabel('Literacy Rate (%)')
axes[0, 0].set_ylabel('Frequency')

axes[0, 1].hist(df_analysis['avg_income_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='lightgreen')
axes[0, 1].set_title('Average Income 2022 Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[0, 1].set_xlabel('Average Income (BRL)')
axes[0, 1].set_ylabel('Frequency')

axes[1, 0].hist(df_analysis['population_2022'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='salmon')
axes[1, 0].set_title('Population 2022 Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[1, 0].set_xlabel('Population')
axes[1, 0].set_ylabel('Frequency')

axes[1, 1].hist(df_analysis['log_population'].dropna(), bins=40, edgecolor='black', alpha=0.7, color='plum')
axes[1, 1].set_title('Log(Population) Distribution (by municipality)', fontsize=12, fontweight='bold')
axes[1, 1].set_xlabel('Log(Population)')
axes[1, 1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## 6. Regional Analysis

In [ ]:
# Regional aggregation from MUNICIPALITY-level data, using POPULATION-WEIGHTED
# statistics. A simple groupby('region_name').agg('mean') would treat every
# municipality equally, which is statistically misleading: tiny towns would
# dominate the average for rates and monetary values.
# For rates and averages across a region we therefore aggregate from totals
# and weight by municipal population.

def _region_rollup(g: pd.DataFrame) -> pd.Series:
    pop = g['population_2022']
    lit_mask = g['literacy_rate_2022'].notna()
    inc_mask = g['avg_income_2022'].notna()
    return pd.Series({
        'N Municipalities': len(g),
        'N States': g['state_code'].nunique(),
        'Total Population': int(pop.sum()),
        'Total Sanctions': int(g['n_sanctions'].sum()),
        'Sanctions/100k (pop-weighted)': round(g['n_sanctions'].sum() / pop.sum() * 100_000, 2),
        'Literacy % (pop-weighted)': round(np.average(g.loc[lit_mask, 'literacy_rate_2022'], weights=pop[lit_mask]), 2),
        'Avg Income BRL (pop-weighted)': round(np.average(g.loc[inc_mask, 'avg_income_2022'], weights=pop[inc_mask]), 2),
    })

regional_summary = (
    df_analysis.groupby('region_name', observed=True)
    .apply(_region_rollup)
)

regional_summary

In [ ]:
# Regional dashboard built from municipality-level data (pop-weighted rates).
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=('Sanctions per 100k by Region (pop-weighted)',
                    'Literacy Rate by Region (pop-weighted)',
                    'Average Income by Region (pop-weighted)',
                    'Total Sanctions by Region'),
    specs=[[{'type': 'bar'}, {'type': 'bar'}],
           [{'type': 'bar'}, {'type': 'bar'}]]
)

regions = regional_summary.reset_index()

fig.add_trace(go.Bar(x=regions['region_name'], y=regions['Sanctions/100k (pop-weighted)'],
                     name='Sanctions/100k', marker_color='indianred'), row=1, col=1)
fig.add_trace(go.Bar(x=regions['region_name'], y=regions['Literacy % (pop-weighted)'],
                     name='Literacy %', marker_color='lightseagreen'), row=1, col=2)
fig.add_trace(go.Bar(x=regions['region_name'], y=regions['Avg Income BRL (pop-weighted)'],
                     name='Income', marker_color='lightsalmon'), row=2, col=1)
fig.add_trace(go.Bar(x=regions['region_name'], y=regions['Total Sanctions'],
                     name='Total Sanctions', marker_color='mediumpurple'), row=2, col=2)

fig.update_layout(height=800, showlegend=False, title_text="Regional Comparison Dashboard (built from 5,570 municipalities)")
fig.show()

## 7. State-Level Rollup and Municipality Extremes

We look at both ends of the granularity spectrum:

1. **State-level rollup** (population-weighted from the 5,570 municipalities) for a stable, policy-relevant bar chart.
2. **Top / bottom municipalities** by `sanctions_per_100k`, which can surface individual outliers that the state rollup hides.

In [ ]:
# Top 10 and bottom 10 MUNICIPALITIES by sanctions per 100k.
# NOTE: per-capita rates for very small municipalities can be unstable
# (denominator effect) -- use with care.
cols = ['municipality_code', 'municipality_name', 'state_name', 'region_name',
        'population_2022', 'n_sanctions', 'sanctions_per_100k']

top_10_munis = df_analysis.nlargest(10, 'sanctions_per_100k')[cols]
bottom_10_munis = df_analysis.nsmallest(10, 'sanctions_per_100k')[cols]

print("TOP 10 MUNICIPALITIES - Highest Sanctions per 100k")
print("=" * 90)
print(top_10_munis.to_string(index=False))

print("\n\nBOTTOM 10 MUNICIPALITIES - Lowest Sanctions per 100k (among munis with sanctions > 0)")
print("=" * 90)
with_sanctions = df_analysis[df_analysis['n_sanctions'] > 0]
print(with_sanctions.nsmallest(10, 'sanctions_per_100k')[cols].to_string(index=False))

In [ ]:
# State-level rollup from municipality data (population-weighted).
# state_name is used on the x-axis (qualitative label); state_code is just an id.
state_rollup = (
    df_analysis.groupby(['state_code', 'state_name', 'region_name'], observed=True)
    .apply(lambda g: pd.Series({
        'population': g['population_2022'].sum(),
        'n_sanctions': g['n_sanctions'].sum(),
        'sanctions_per_100k': g['n_sanctions'].sum() / g['population_2022'].sum() * 100_000,
    }))
    .reset_index()
    .sort_values('sanctions_per_100k', ascending=False)
)

fig = px.bar(state_rollup,
             x='state_name', y='sanctions_per_100k',
             color='region_name',
             title='Sanctions per 100k Population by State (rolled up from 5,570 municipalities)',
             labels={'sanctions_per_100k': 'Sanctions per 100k', 'state_name': 'State'},
             height=500)
fig.update_xaxes(tickangle=-45)
fig.show()

## 8. Sanctions Registry Analysis

In [ ]:
if df_sanctions is not None:
    print("Sanctions by Registry Type:")
    print("=" * 60)
    display(df_sanctions[['registry_type', 'total_sanctions', 'sanctions_pf', 'sanctions_pj', 'pj_ratio_pct']])
    
    fig = px.pie(df_sanctions, values='total_sanctions', names='registry_type',
                 title='Sanctions Distribution by Registry Type')
    fig.show()


## 9. Correlation Heatmap (Preview)

In [ ]:
# Correlation matrix of key analytical features at municipality level
# (5,570 observations instead of 27 states -- much more statistical power).
# We include the region dummies but NOT the 27 state dummies (would make the
# heatmap unreadable). Transfer-side features are also included since the
# thesis question links federal transfers to compliance outcomes.

corr_cols = ['sanctions_per_100k', 'literacy_rate_2022', 'avg_income_2022',
             'log_population', 'log_income']
# Include log_total_transfers if present (new in muni dataset)
if 'log_total_transfers' in df_analysis.columns:
    corr_cols.append('log_total_transfers')
corr_cols += REGION_DUMMY_COLS

# Cast to float64 (numpy) so np.corrcoef / seaborn are happy with pandas
# nullable Int64/Float64 + rows that contain NaN in any included column.
corr_df = df_analysis[corr_cols].astype('Float64').astype(float)
corr_matrix = corr_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlation Matrix: Key Variables (municipality-level, N=5,570)',
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Key Findings Summary

### 10.1 Data Quality
- **Grain:** this EDA runs at **municipality level** (5,570 rows, 1 per Brazilian municipality), up from the previous state-level view (27 rows). This gives ~200x more statistical power for correlation and regression work downstream.
- **Complete geographic coverage**: all 27 states and all 5 regions are represented.
- **Sanctions data is dense**: `n_sanctions` is non-null for all 5,570 municipalities (zeros are real, not missing).
- **Transfer-rate feature is sparse**: `sanctions_per_million_brl_transfers` is null for ~91.5% of municipalities (most have no federal transfer records in the current Gold cut). Use with care in any modeling that depends on it.

### 10.2 Regional Patterns (population-weighted)
- Regional numbers are computed as `sum(sanctions) / sum(population) * 100_000` across each region's municipalities -- not as an unweighted mean of per-muni rates -- so they are not dominated by tiny municipalities.
- Once weighted correctly, the ordering of regions by sanctions/100k is flatter than the old state-level unweighted view suggested; see the regional summary table above for the actual values on the current Gold snapshot.

### 10.3 State and Municipality Extremes
- The state-level bar chart is now computed as a **rollup from municipality data** (population-weighted), so every state's number is consistent with the regional totals.
- The top-10 municipalities by `sanctions_per_100k` surface individual outliers that the state rollup hides. Rates for very small municipalities can be unstable (denominator effect) and should be interpreted alongside absolute `n_sanctions` and `population_2022`.

### 10.4 Dummy variables
- Region and state are preserved as identifier columns (`state_code`, `state_name`, `region_code`, `region_name`) AND as one-hot dummies (`is_region_*`, `is_state_*`) for use as regression features.
- The mean of a dummy equals the proportion of municipalities in that category (e.g. `df['is_region_1'].mean()` == share of munis in the Norte region). See the dummy summary table for the actual shares.

### 10.5 Correlations (municipality-level)
- With N=5,570 the correlation coefficients in the heatmap are far more reliable than at the 27-state grain. Inspect the `sanctions_per_100k` row / column in the heatmap above for the strongest bivariate signals; the thesis question about federal transfers vs. compliance outcomes can now be tested at the unit of observation where policy actually lands (the municipality).